In [1]:
from utilities.utils import SSPModelForCalibration, HelperFunctions
from utilities.diff_reports import DiffReportUtils
from ssp_transformations_handler.GeneralUtils import GeneralUtils
import pandas as pd
import warnings
import os
import numpy as np
warnings.filterwarnings("ignore")

##  IMPORT SISEPUEDE EXAMPLES AND TRANSFORMERS

from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
import sisepuede.core.support_classes as sc
import sisepuede.transformers as trf
import sisepuede.utilities._plotting as sup
import sisepuede.utilities._toolbox as sf

# temporary fix to avoid rebuilding logger everytime
log_job = None

/home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/munch/__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
gu = GeneralUtils()

In [4]:
# Define paths
SCRIPTS_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPTS_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, 'data')
OUTPUT_DIR_PATH = os.path.join(ROOT_DIR_PATH, 'output')
MISC_DIR_PATH = os.path.join(SCRIPTS_DIR_PATH, 'misc')
DUMMY_DIR_PATH = os.path.join(MISC_DIR_PATH, 'dummy')
SECTORAL_REPORT_MAPPING_DIR_PATH = os.path.join(MISC_DIR_PATH, 'sectoral_report_mapping')
SECTORAL_REPORTS_DIR_PATH = os.path.join(MISC_DIR_PATH, 'sectoral_reports')

In [5]:
# Define parameters
iso_alpha_3 = 'UGA'
region = 'uganda'
energy_model_flag = True
# run_id = '20250605161834'
# use_edgar_db_flag = False
ssp_edgar_cw_file_name = 'sisepuede_edgar_active_crosswalk.csv'
# RUN_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, region, run_id)
INPUT_DATA_PATH = os.path.join(DATA_DIR_PATH, "new_uganda_inputs.csv")

## Create a SSP output file

In [6]:
input_df = pd.read_csv(INPUT_DATA_PATH)
input_df.head()

,region,time_period,area_gnrl_country_ha,avgload_trns_freight_tonne_per_vehicle_aviation,avgload_trns_freight_tonne_per_vehicle_rail_freight,avgload_trns_freight_tonne_per_vehicle_road_heavy_freight,avgload_trns_freight_tonne_per_vehicle_water_borne,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,...,yf_agrc_fruits_tonne_ha,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha
0,uganda,0,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,7.373499,2.123306,0.762027,0.982737,0.3674,0.784492,2.755775,76.794300,4.446020,4.002223
1,uganda,1,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,7.209806,2.171183,0.716171,0.984473,0.3674,0.803826,2.826323,77.459654,4.558346,4.070985
2,uganda,2,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,7.187318,2.136681,0.771066,0.999795,0.3674,0.808390,2.908776,77.263223,5.202683,4.007029
3,uganda,3,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,7.173246,2.134035,0.744831,1.441325,0.3674,0.804774,2.961356,76.784930,5.605283,4.014348
4,uganda,4,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,7.219285,2.172065,0.458120,1.528325,0.3674,0.799814,2.970174,77.726965,4.805888,4.035314


In [7]:
# Load input dataset
examples = SISEPUEDEExamples()
cr = examples("input_data_frame")
cr.head()

,region,time_period,avgload_trns_freight_tonne_per_vehicle_aviation,avgload_trns_freight_tonne_per_vehicle_rail_freight,avgload_trns_freight_tonne_per_vehicle_road_heavy_freight,avgload_trns_freight_tonne_per_vehicle_water_borne,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,avgmass_lvst_animal_chickens_kg,...,ef_lndu_conv_wetlands_to_forests_mangroves_gg_co2_ha,ef_lndu_conv_wetlands_to_forests_primary_gg_co2_ha,ef_lndu_conv_wetlands_to_forests_secondary_gg_co2_ha,ef_lndu_conv_wetlands_to_grasslands_gg_co2_ha,ef_lndu_conv_wetlands_to_other_gg_co2_ha,ef_lndu_conv_wetlands_to_pastures_gg_co2_ha,ef_lndu_conv_wetlands_to_settlements_gg_co2_ha,ef_lndu_conv_wetlands_to_shrublands_gg_co2_ha,ef_lndu_conv_wetlands_to_wetlands_gg_co2_ha,gasrf_lsmm_biogas_anaerobic_lagoon
0,costa_rica,0,70.0,2923.0,31.751466,6468.0,322.900664,520.741388,310.599686,1.12759,...,0.0,0.0,0.0,0.0,0.003030,0.0,0.003030,0.0,0.0,0.0
1,costa_rica,1,70.0,2923.0,31.751466,6468.0,322.900664,520.741388,310.599686,1.12759,...,0.0,0.0,0.0,0.0,0.003010,0.0,0.003010,0.0,0.0,0.0
2,costa_rica,2,70.0,2923.0,31.751466,6468.0,322.900664,520.741388,310.599686,1.12759,...,0.0,0.0,0.0,0.0,0.002960,0.0,0.002960,0.0,0.0,0.0
3,costa_rica,3,70.0,2923.0,31.751466,6468.0,322.900664,520.741388,310.599686,1.12759,...,0.0,0.0,0.0,0.0,0.002949,0.0,0.002949,0.0,0.0,0.0
4,costa_rica,4,70.0,2923.0,31.751466,6468.0,322.900664,520.741388,310.599686,1.12759,...,0.0,0.0,0.0,0.0,0.002949,0.0,0.002949,0.0,0.0,0.0


In [8]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
gu.compare_dfs(cr, input_df)

Columns in df_example but not in df_input: {'pij_lndu_flooded_to_forests_mangroves', 'ef_lndu_conv_shrublands_to_shrublands_gg_co2_ha', 'pij_lndu_forests_secondary_to_pastures', 'area_lndu_infimum_wetlands_ha', 'pij_lndu_pastures_to_settlements', 'pij_lndu_other_to_flooded', 'pij_lndu_flooded_to_croplands', 'area_lndu_infimum_settlements_ha', 'pij_lndu_flooded_to_forests_primary', 'pij_lndu_other_to_shrublands', 'ef_lndu_sequestration_pastures_kt_co2_ha', 'ef_lndu_conv_other_to_pastures_gg_co2_ha', 'factor_lndu_soil_carbon_shrublands', 'ef_lndu_conv_forests_secondary_to_pastures_gg_co2_ha', 'frac_lndu_improved_pastures', 'ef_lndu_conv_shrublands_to_pastures_gg_co2_ha', 'pij_lndu_pastures_to_forests_primary', 'ef_lndu_conv_flooded_to_forests_secondary_gg_co2_ha', 'ef_lndu_conv_shrublands_to_settlements_gg_co2_ha', 'frac_lndu_increasing_net_exports_met_pastures', 'pij_lndu_flooded_to_flooded', 'pij_lndu_shrublands_to_pastures', 'ef_lndu_conv_grasslands_to_shrublands_gg_co2_ha', 'pij_lndu

In [9]:
# Add missing columns and reformat the input datas
input_df_complete = input_df.rename(columns={'period': 'time_period'})
input_df_complete = gu.add_missing_cols(cr, input_df_complete.copy())
input_df_complete = input_df_complete.drop(columns='iso_code3', errors='ignore')

# Subset input_df_complete to the input rows amount
input_df_complete.head()

,region,time_period,area_gnrl_country_ha,avgload_trns_freight_tonne_per_vehicle_aviation,avgload_trns_freight_tonne_per_vehicle_rail_freight,avgload_trns_freight_tonne_per_vehicle_road_heavy_freight,avgload_trns_freight_tonne_per_vehicle_water_borne,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,...,ef_lndu_conv_shrublands_to_grasslands_gg_co2_ha,ef_lndu_conv_shrublands_to_other_gg_co2_ha,ef_lndu_conv_shrublands_to_pastures_gg_co2_ha,ef_lndu_conv_shrublands_to_settlements_gg_co2_ha,ef_lndu_conv_shrublands_to_shrublands_gg_co2_ha,ef_lndu_conv_shrublands_to_wetlands_gg_co2_ha,ef_lndu_conv_wetlands_to_flooded_gg_co2_ha,ef_lndu_conv_wetlands_to_pastures_gg_co2_ha,ef_lndu_conv_wetlands_to_shrublands_gg_co2_ha,gasrf_lsmm_biogas_anaerobic_lagoon
0,uganda,0,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049159,0.052189,0.049159,0.052189,0.0,0.049159,0.003030,0.0,0.0,0.0
1,uganda,1,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049179,0.052189,0.049179,0.052189,0.0,0.049179,0.003010,0.0,0.0,0.0
2,uganda,2,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049229,0.052189,0.049229,0.052189,0.0,0.049229,0.002960,0.0,0.0,0.0
3,uganda,3,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049239,0.052189,0.049239,0.052189,0.0,0.049239,0.002949,0.0,0.0,0.0
4,uganda,4,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049239,0.052189,0.049239,0.052189,0.0,0.049239,0.002949,0.0,0.0,0.0


In [10]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
gu.compare_dfs(cr, input_df_complete)

Columns in df_example but not in df_input: set()
Columns in df_input but not in df_example: set()


In [11]:
input_df_complete = input_df_complete[(input_df_complete['time_period'] > 4) & (input_df_complete['time_period'] <= 7)].reset_index(drop=True)
input_df_complete

,region,time_period,area_gnrl_country_ha,avgload_trns_freight_tonne_per_vehicle_aviation,avgload_trns_freight_tonne_per_vehicle_rail_freight,avgload_trns_freight_tonne_per_vehicle_road_heavy_freight,avgload_trns_freight_tonne_per_vehicle_water_borne,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,...,ef_lndu_conv_shrublands_to_grasslands_gg_co2_ha,ef_lndu_conv_shrublands_to_other_gg_co2_ha,ef_lndu_conv_shrublands_to_pastures_gg_co2_ha,ef_lndu_conv_shrublands_to_settlements_gg_co2_ha,ef_lndu_conv_shrublands_to_shrublands_gg_co2_ha,ef_lndu_conv_shrublands_to_wetlands_gg_co2_ha,ef_lndu_conv_wetlands_to_flooded_gg_co2_ha,ef_lndu_conv_wetlands_to_pastures_gg_co2_ha,ef_lndu_conv_wetlands_to_shrublands_gg_co2_ha,gasrf_lsmm_biogas_anaerobic_lagoon
0,uganda,5,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049209,0.052189,0.049209,0.052189,0.0,0.049209,0.00298,0.0,0.0,0.0
1,uganda,6,24155000.0,70.0,4082.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049209,0.052189,0.049209,0.052189,0.0,0.049209,0.00298,0.0,0.0,0.0
2,uganda,7,24155000.0,70.0,4082.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049209,0.052189,0.049209,0.052189,0.0,0.049209,0.00298,0.0,0.0,0.0


In [12]:
# Initialize the SSP model
ssp_model = SSPModelForCalibration(energy_model_flag=energy_model_flag)

2025-07-31 14:22:46,567 - INFO - Successfully initialized SISEPUEDEFileStructure.
2025-07-31 14:22:46,570 - WARNING - Missing key dict_dimensional_keys: key time_series not found. Tables that rely on the time_series will not have index checking.
2025-07-31 14:22:46,571 - INFO - 	Setting export engine to 'csv'.
2025-07-31 14:22:46,572 - WARNING - No index fields defined. Index field values will not be checked when writing to tables.
2025-07-31 14:22:46,572 - INFO - Successfully instantiated table ANALYSIS_METADATA
2025-07-31 14:22:46,573 - WARNING - No index fields found in ATTRIBUTE_DESIGN. Initializing index fields.
2025-07-31 14:22:46,573 - INFO - Successfully instantiated table ATTRIBUTE_DESIGN
2025-07-31 14:22:46,574 - WARNING - No index fields found in ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES. Initializing index fields.
2025-07-31 14:22:46,575 - INFO - Successfully instantiated table ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES
2025-07-31 14:22:46,575 - WARNING - No index fi

yay


2025-07-31 14:22:47,849 - INFO - Initializing FutureTrajectories
2025-07-31 14:22:52,332 - INFO - Instantiating 1392 sampling units.
2025-07-31 14:22:52,370 - INFO - Iteration 0 complete.
2025-07-31 14:23:02,831 - INFO - Iteration 250 complete.
2025-07-31 14:23:07,604 - INFO - Iteration 500 complete.
2025-07-31 14:23:12,144 - INFO - Iteration 750 complete.
2025-07-31 14:23:16,905 - INFO - Iteration 1000 complete.
2025-07-31 14:23:21,653 - INFO - Iteration 1250 complete.
2025-07-31 14:23:24,220 - INFO - 	1392 sampling units complete in 31.89 seconds.
2025-07-31 14:23:24,228 - INFO - 	FutureTrajectories for 'costa_rica' complete.
2025-07-31 14:23:24,229 - INFO - Initializing LHSDesign
2025-07-31 14:23:24,230 - INFO - LHSDesign.fields_factors_l reset successful.
2025-07-31 14:23:24,230 - INFO - LHSDesign.fields_factors_x reset successful.
2025-07-31 14:23:24,242 - INFO - 	LHSDesign for region 'costa_rica' complete.
2025-07-31 14:23:24,244 - INFO - Generating primary keys (values of primar

[juliapkg] Found dependencies: /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/julia/pyjuliapkg/juliapkg.json
[juliapkg] Locating Julia ^1.11.5
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.6 to /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/julia/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia f

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `~/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/julia/Project.toml`
  [9961bab8] + Cbc v1.2.0
  [e2554f3b] + Clp v1.2.2
  [a93c6f00] + DataFrames v1.7.0
  [60bf3e95] + GLPK v1.2.1
  [87dc4568] + HiGHS v1.19.0
  [b6b21f68] + Ipopt v1.10.6
  [4076af6c] + JuMP v1.27.0
  [a3c327a0] + NemoMod v2.0.0 `https://github.com/sei-international/NemoMod.jl.git#61e63e0`
⌅ [6099a3de] + PythonCall v0.9.22
  [0aa819cd] + SQLite v1.6.1
    Updating `~/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/julia/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.6.0
  [9961bab8] + Cbc v1.2.0
  [e2554f3b] + Clp v1.2.2
  [523fee87] + CodecBzip2 v0.8.5
  [944b1d66] + CodecZlib v0.7.8
  [bbf7d656] + CommonSubexpressions v0.3.1
  [34da2185] + Compat v4.18.0
  [992eb4ea] + CondaPkg v0.2.29
  [88353bc9] + ConfParser v0.1.2
  [a8cc5b0e] + Crayons v4.1.1
  [a10d1c49] + DBInterface v2.6.1
 

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Precompiling NemoMod...
Info Given NemoMod was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   2260.1 ms  ? NemoMod
[ Info: Precompiling NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72] 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
2025-07-31 14:30:52,944 - INFO - Successfully initialized JuMP optimizer from solver module HiGHS.
2025-07-31 14:30:53,001 - INFO - Successfully initialized SISEPUEDEModels.
2025-07-31 14:30:53,022 - INFO - Table ANALYSIS_METADATA successfully written to /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11

In [13]:
output_df = ssp_model.run_ssp_simulation(input_df_complete)

2025-07-31 14:30:53,770 - INFO - Running AFOLU model
2025-07-31 14:30:53,937 - INFO - AFOLU model run successfully completed
2025-07-31 14:30:53,938 - INFO - Running CircularEconomy model
2025-07-31 14:30:53,994 - INFO - CircularEconomy model run successfully completed
2025-07-31 14:30:53,994 - INFO - Running IPPU model
2025-07-31 14:30:54,074 - INFO - IPPU model run successfully completed
2025-07-31 14:30:54,075 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2025-07-31 14:30:54,100 - DEBUG - Missing elasticity information found in 'project_energy_consumption_by_fuel_from_effvars': using specified future demands.
2025-07-31 14:30:54,188 - INFO - EnergyConsumption without Fugitive Emissions model run successfully completed
2025-07-31 14:30:54,189 - INFO - Running Energy model (Electricity and Fuel Production: trying to call Julia)
2025-07-31 14:30:54,261 - INFO - 	Path to temporary NemoMod database '/home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/si

2025-31-Jul 14:30:55.048 Opened SQLite database at /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/tmp/nemomod_intermediate_database.sqlite.
2025-31-Jul 14:30:55.469 Added NEMO structure to SQLite database at /home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/sisepuede/tmp/nemomod_intermediate_database.sqlite.


2025-31-Jul 14:31:21.240 Started modeling scenario. NEMO version = 2.0.0, solver = HiGHS.


┌ Warning: Model period emission limits (ModelPeriodEmissionLimit parameter) are not enforced when modeling selected years.
└ @ NemoMod ~/.julia/packages/NemoMod/p49Bn/src/scenario_calculation.jl:6112


2025-31-Jul 14:32:07.948 Finished modeling scenario.


2025-07-31 14:32:08,321 - INFO - NemoMod ran successfully with the following status: OPTIMAL
2025-07-31 14:32:08,342 - INFO - EnergyProduction model run successfully completed
2025-07-31 14:32:08,343 - INFO - Running Energy (Fugitive Emissions)
2025-07-31 14:32:08,397 - INFO - Fugitive Emissions from Energy model run successfully completed
2025-07-31 14:32:08,399 - INFO - Appending Socioeconomic outputs
2025-07-31 14:32:08,410 - INFO - Socioeconomic outputs successfully appended.


In [42]:
output_df.head()

,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,area_agrc_crops_other_woody_perennial,area_agrc_crops_pulses,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,0,747901.028486,2.386299e+06,183833.867477,1.333131e+06,595.329750,605228.344795,3.433943e+06,4242.254753,1.090824e+06,...,9.829842e+06,1264.067389,461200.281192,3.374662e+06,1558.604070,855742.438638,371284.928361,8.144525e+06,8.463351e+06,2.839233e+06
1,1,746180.919732,2.380811e+06,183411.065217,1.330065e+06,593.960543,603836.371078,3.426045e+06,4232.497928,1.088315e+06,...,9.589512e+06,1289.597171,432449.975084,3.372849e+06,1555.019391,874816.334236,379914.032975,8.196196e+06,8.657214e+06,2.881371e+06
2,2,745855.348591,2.379772e+06,183331.039920,1.329485e+06,593.701388,603572.907231,3.424550e+06,4230.651219,1.087840e+06,...,9.555432e+06,1268.550416,465394.440892,3.423848e+06,1554.340889,879399.053014,390826.724560,8.171843e+06,9.876628e+06,2.834867e+06
3,3,746831.986263,2.382888e+06,183571.097192,1.331226e+06,594.478792,604363.237474,3.429034e+06,4236.190917,1.089264e+06,...,9.549210e+06,1268.638786,450148.390655,4.942354e+06,1556.376205,876612.253332,398412.478937,8.131891e+06,1.065485e+07,2.843764e+06
4,4,748702.931966,2.388858e+06,184030.975132,1.334561e+06,595.968066,605877.273861,3.437624e+06,4246.803321,1.091993e+06,...,9.634575e+06,1294.481423,277564.514174,5.253808e+06,1560.275189,873391.596641,400599.954899,8.252278e+06,9.158196e+06,2.865777e+06


In [43]:
if energy_model_flag:
    output_df.to_csv(f'misc/dummy/ssp_{region}_output_dummy_energy.csv', index=False)

else:
    output_df.to_csv(f'misc/dummy/ssp_{region}_output_dummy.csv', index=False)